# Attention: fused and decomposed, and where the relevance goes

`scaled_dot_product_attention` can be handled two ways. By default autoLRP rewrites it during the forward into matmul, scale, softmax, matmul, so every step is an ordinary node with an ordinary rule. With `set_decompose_attention(False)` the fused kernel stays, and its single backward node is handled by one installer that reconstructs the attention matrix and runs the same rules. This notebook checks the two agree, shows what the presets do to the three inputs, and looks at cross-attention with a constant memory.

In [1]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
import torch
import torch.nn.functional as F

import autoLRP
from autoLRP import LRPConfig, BASE, set_decompose_attention, explain, explain_summary
torch.manual_seed(0)

d, heads_q, heads_kv, T = 16, 4, 2, 6              # grouped-query attention: 4 query heads share 2 key/value heads
Q = torch.randn(1, heads_q, T, d, dtype=torch.float64)
K = torch.randn(1, heads_kv, T, d, dtype=torch.float64)
V = torch.randn(1, heads_kv, T, d, dtype=torch.float64)

def attend(q, k, v):
    return F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)

def run(decomposed, config):
    set_decompose_attention(decomposed)
    try:
        q, k, v = autoLRP.tensor(Q.clone()), autoLRP.tensor(K.clone()), autoLRP.tensor(V.clone())
        attend(q, k, v).sum().lrp(config=config)
    finally:
        set_decompose_attention(True)
    return q.relevance, k.relevance, v.relevance

## The two paths agree

Float64, causal mask, grouped-query heads: the largest difference between the fused and the decomposed path is at the level of rounding.

In [2]:
for name, cfg in [('cplrp', LRPConfig(attn='cplrp')), ('attnlrp', LRPConfig(attn='attnlrp')), ('uniform', LRPConfig(attn='uniform')),
                  ('BASE', LRPConfig())]:
    dec, fus = run(True, cfg), run(False, cfg)
    diffs = [float((a - b).abs().max()) for a, b in zip(dec, fus)]
    print(f'{name:8s} max |fused - decomposed| on q, k, v = {diffs[0]:.1e}, {diffs[1]:.1e}, {diffs[2]:.1e}')

cplrp    max |fused - decomposed| on q, k, v = 0.0e+00, 0.0e+00, 1.3e-17
attnlrp  max |fused - decomposed| on q, k, v = 2.4e-13, 2.8e-13, 6.1e-18
uniform  max |fused - decomposed| on q, k, v = 2.4e-12, 2.3e-12, 6.9e-18
BASE     max |fused - decomposed| on q, k, v = 9.5e-12, 9.2e-12, 6.1e-18


## What each preset does with the three inputs

The output relevance seeded here is 1.0 in total. `cplrp` treats the attention weights as constants: everything goes to the values. `BASE` (epsilon on both products, passthrough softmax) halves at the `A @ V` product and again at `Q @ K^T`, and conserves. `uniform` is gradient times input, halved per operand, and does not conserve; `attnlrp` propagates through the softmax with its Jacobian, which does not conserve either.

In [3]:
for name, cfg in [('cplrp', LRPConfig(attn='cplrp')), ('BASE', LRPConfig()), ('uniform', LRPConfig(attn='uniform')), ('attnlrp', LRPConfig(attn='attnlrp'))]:
    rq, rk, rv = run(True, cfg)
    print(f'{name:8s} R_q = {float(rq.sum()):+.3f}   R_k = {float(rk.sum()):+.3f}   R_v = {float(rv.sum()):+.3f}   total = {float(rq.sum() + rk.sum() + rv.sum()):+.3f}')

cplrp    R_q = +0.000   R_k = +0.000   R_v = +1.000   total = +1.000
BASE     R_q = +0.250   R_k = +0.250   R_v = +0.500   total = +1.000
uniform  R_q = +0.023   R_k = +0.023   R_v = +0.097   total = +0.142
attnlrp  R_q = +0.045   R_k = +0.045   R_v = +0.500   total = +0.590


## Reading the fused node

Decomposed, the two products are two `BmmBackward0` nodes; fused, they are one node that reports two lines, resolved separately, with the same keys as the decomposed graph. The `1/sqrt(d)` scale is a multiplication by a constant and passes relevance through unchanged in both.

In [4]:
cfg = LRPConfig(attn='cplrp')
q, k, v = autoLRP.tensor(Q.clone()), autoLRP.tensor(K.clone()), autoLRP.tensor(V.clone())
print('decomposed')
print(explain_summary([r for r in explain(attend(q, k, v).sum(), cfg) if r[2] != 'native gradient']))
set_decompose_attention(False)
try:
    q, k, v = autoLRP.tensor(Q.clone()), autoLRP.tensor(K.clone()), autoLRP.tensor(V.clone())
    print()
    print('fused')
    print(explain_summary([r for r in explain(attend(q, k, v).sum(), cfg) if r[2] != 'native gradient']))
finally:
    set_decompose_attention(True)

decomposed
count  node             key                   what
    1  AddBackward      AddBackward           residual_proportional
    1  BmmBackward0     weights_operand       detach_lhs_bmm
    1  BmmBackward0     BmmBackward           epsilon_bmm
    1  MulBackward0     None                  passthrough (constant operand)
    1  SoftmaxBackward  None                  softmax=passthrough
    1  SumBackward      None                  reduction_share

fused
count  node                                           key                   what
    1  ScaledDotProductFlashAttentionForCpuBackward0  weights_operand       detach_lhs_bmm
    1  ScaledDotProductFlashAttentionForCpuBackward0  BmmBackward           epsilon_bmm
    1  ScaledDotProductFlashAttentionForCpuBackward0  None                  softmax=passthrough
    1  SumBackward                                    None                  reduction_share


## Cross-attention with a constant memory

When the keys and values are not wrapped (a fixed memory, an encoder output you do not attribute to), each product has one operand from the input and is a linear layer whose weight is the other operand. All relevance then reaches the query, under every preset that conserves.

In [5]:
def run_memory(decomposed, config):
    set_decompose_attention(decomposed)
    try:
        q = autoLRP.tensor(Q.clone())
        attend(q, K.clone(), V.clone()).sum().lrp(config=config)
    finally:
        set_decompose_attention(True)
    return q.relevance

for name, cfg in [('cplrp', LRPConfig(attn='cplrp')), ('BASE', LRPConfig()), ('uniform', LRPConfig(attn='uniform'))]:
    dec, fus = run_memory(True, cfg), run_memory(False, cfg)
    print(f'{name:8s} R_q = {float(dec.sum()):+.3f}   fused equals decomposed to {float((dec - fus).abs().max()):.1e}')

cplrp    R_q = +1.000   fused equals decomposed to 3.8e-11
BASE     R_q = +1.000   fused equals decomposed to 3.8e-11
uniform  R_q = +1.000   fused equals decomposed to 3.8e-11
